# Visualization

Цей ноутбук будує графіки:
- Стовпчикова діаграма кількості документів по кластерах
- Теплова карта інтенсивності за темами та днями
- PCA-візуалізація кластерів
- Зберігає графіки у `/shared/plots/`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

db_host = os.environ.get('MYSQL_HOST', 'db')
db_user = os.environ.get('MYSQL_USER', 'appuser')
db_password = os.environ.get('MYSQL_PASSWORD', 'apppassword')
db_name = os.environ.get('MYSQL_DATABASE', 'docflow')

connection_string = f'mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/{db_name}'
engine = create_engine(connection_string)

df = pd.read_sql('SELECT * FROM documents', engine)
print(f'[visualization] Завантажено {len(df)} записів')

plots_dir = '/shared/plots'
os.makedirs(plots_dir, exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
text_col = None
for col in df.columns:
    if 'зміст' in col.lower() or 'короткий' in col.lower():
        text_col = col
        break

if text_col:
    df[text_col] = df[text_col].fillna('немає змісту').astype(str)
    vectorizer = TfidfVectorizer(max_features=500, stop_words=None)
    X = vectorizer.fit_transform(df[text_col])

    num_clusters = 4
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)
    print(f'Кластеризація виконана: {num_clusters} кластерів')
else:
    print('УВАГА: текстовий стовпець не знайдено!')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df, x='cluster', hue='cluster',
    palette='viridis', legend=False
)
plt.title('Кількість документів у кожному кластері', fontsize=15)
plt.xlabel('Номер кластера (теми)', fontsize=12)
plt.ylabel('Кількість документів', fontsize=12)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 9),
                textcoords='offset points')
plt.tight_layout()
plt.savefig(f'{plots_dir}/cluster_distribution.png', dpi=150)
plt.close()
print('Графік 1 збережено: cluster_distribution.png')

In [ ]:
date_col = None
for col in df.columns:
    if 'дата' in col.lower() or 'реєстрац' in col.lower():
        date_col = col
        break

if date_col:
    df[date_col] = pd.to_datetime(df[date_col])
    pivot_table = df.pivot_table(
        index='cluster',
        columns=df[date_col].dt.day,
        values=df.columns[0],
        aggfunc='count'
    ).fillna(0)

    plt.figure(figsize=(14, 6))
    sns.heatmap(pivot_table, annot=True, cmap='YlGnBu', fmt='g')
    plt.title('Інтенсивність виходу документів за темами (дні місяця)', fontsize=15)
    plt.xlabel('День місяця', fontsize=12)
    plt.ylabel('Номер кластера', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/heatmap_intensity.png', dpi=150)
    plt.close()
    print('Графік 2 збережено: heatmap_intensity.png')

In [ ]:
if text_col:
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X.toarray())

    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        x=coords[:, 0], y=coords[:, 1],
        hue=df['cluster'],
        palette='viridis', alpha=0.7
    )
    plt.title('Геометричне розділення кластерів (PCA)', fontsize=15)
    plt.xlabel('PC1', fontsize=12)
    plt.ylabel('PC2', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/pca_clusters.png', dpi=150)
    plt.close()
    print('Графік 3 збережено: pca_clusters.png')

print(f'\nУсі графіки збережено в {plots_dir}')